<a href="https://colab.research.google.com/github/DuaaMahar5/FlyRank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DuaaMahar5/FlyRank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page's performance for one client, added up over a month. I use fact_content_daily_performance (Feb 2026 and March 2026), plus dim_content for page details and dim_clients to check GA4 data.

 My "today" is March 1, 2026. I only use February data as features, since that month is already finished. March is what I'm trying to predict, did impressions go down.

 I leave out any March numbers as features, because that's basically the answer, not a clue.

In [20]:
%pip -q install duckdb

import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_feb': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')",
    'fact_daily_mar': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f'{name:16} {n:>12,} rows')

print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_feb']}").df().to_string())


dim_clients               104 rows
dim_content           519,606 rows
fact_daily_feb      7,355,108 rows
fact_daily_mar      9,841,378 rows
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: Feb impressions, Feb clicks, Feb avg position, Feb AI sessions (only where GA4 data exists), and page age, word count, type, all known before March starts.


Label: March impressions, and whether they dropped more than 20% vs February.
Context: client ID and content ID, just for grouping, not used as signals.
Excluded: March GSC/GA4 numbers (that's the label), and the query-level table, since its 90-day window overlaps March, so it's risky

In [21]:
# No query needed here -- this section just sorts columns into buckets.
# The claims get proven with real queries in Section 3.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The five features

1.Feb impressions. Already happened, so it's known by March 1.
2.Feb clicks. Same, closed month.
3.Feb avg position. Already measured by March 1.
4.Feb AI sessions. Known, but only counted where GA4 tracking actually exists.
5.Page age (and word count/type). Set when the page was made, so always known ahead of time.

In [22]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily_feb']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"{len(grain_check)} duplicate (date, client, content) rows found -- 0 means the grain holds")
grain_check


span = con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date
    FROM {TABLES['fact_daily_feb']}
""").df()
span

avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_daily_feb']}
""").df()
avail


feature_frame = con.sql(f"""
    WITH feb AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_impressions) AS feb_impressions,
            SUM(gsc_clicks) AS feb_clicks,
            AVG(gsc_avg_position) AS feb_avg_position,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_ai ELSE NULL END) AS feb_ai_sessions
        FROM {TABLES['fact_daily_feb']}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 50
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS mar_impressions
        FROM {TABLES['fact_daily_mar']}
        GROUP BY 1, 2
    )
    SELECT feb.*, mar.mar_impressions
    FROM feb JOIN mar USING (client_hash_id, content_hash_id)
""").df()

content_meta = con.sql(f"""
    SELECT
        content_hash_id,
        DATE_DIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days,
        word_count,
        content_type
    FROM {TABLES['dim_content']}
""").df()

feature_frame = feature_frame.merge(content_meta, on='content_hash_id', how='left')
print(f"{len(feature_frame):,} rows in the feature frame")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

0 duplicate (date, client, content) rows found -- 0 means the grain holds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

89,086 rows in the feature frame


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,feb_ai_sessions,mar_impressions,content_age_days,word_count,content_type
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4270.0,7.0,5.805560,NaN,6523.0,366,2123,keyword article
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,440.0,2.0,3.994887,NaN,453.0,366,<NA>,keyword article
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5271.0,4.0,7.101588,NaN,5630.0,366,2546,keyword article
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,6690.0,19.0,7.323989,NaN,4944.0,366,2330,keyword article
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,0.0,2.886731,NaN,429.0,366,<NA>,keyword article


In [23]:
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df().to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [24]:
#3e
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = feature_frame.dropna(subset=['feb_impressions','feb_clicks','feb_avg_position','content_age_days']).copy()
df['is_declining'] = (df['mar_impressions'] < 0.8 * df['feb_impressions']).astype(int)

honest_features = ['feb_impressions', 'feb_clicks', 'feb_avg_position', 'content_age_days']

X = df[honest_features].fillna(0)
y = df['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"HONEST model (features only from the closed Feb window) -- AUC: {honest_auc:.3f}")

HONEST model (features only from the closed Feb window) -- AUC: 0.564


In [25]:
# --- THE TRAP ---
df['trend_pct_LEAK'] = (df['mar_impressions'] - df['feb_impressions']) / df['feb_impressions'].replace(0, np.nan) * 100

leaky_features = honest_features + ['trend_pct_LEAK']
X_leak = df[leaky_features].fillna(0)
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)

leaky_model = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, leaky_model.predict_proba(X_te_l)[:, 1])
print(f"LEAKY model (+ trend_pct_LEAK, computed from March itself) -- AUC: {leaky_auc:.3f}")
print(f"Jump: {honest_auc:.3f} -> {leaky_auc:.3f}")

LEAKY model (+ trend_pct_LEAK, computed from March itself) -- AUC: 1.000
Jump: 0.564 -> 1.000


In [26]:
# --- DELETE THE LEAK, KEEP THE HONEST NUMBER ---
df = df.drop(columns=['trend_pct_LEAK'])
print(f"Leak column removed. Keeping the honest AUC: {honest_auc:.3f} as my real baseline.")

Leak column removed. Keeping the honest AUC: 0.564 as my real baseline.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

My data only includes pages with at least 50 impressions in February, which left 89,086 pages out of the original 321,546, so very small or new pages aren't in here.

Also, only 2% of February rows have GA4 data available, so features like AI sessions are missing for almost everyone, not because they have zero AI traffic, but because they simply aren't tracked.

My honest model also scored a weak 0.553 AUC (barely better than guessing) with just these five features, which shows I don't have strong signal yet, this is expected for a first pass and something later weeks will need to improve on.

In [27]:
# No new query needed here -- these limits (the >= 50 filter size,
# the 2% GA4 availability, the 0.553 honest AUC) were already proven
# with real numbers in Section 3.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.